In [2]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving Airline_Data_Extracted.xlsx to Airline_Data_Extracted.xlsx


In [3]:
df = pd.read_excel("Airline_Data_Extracted.xlsx")
print(df.shape)
df.head()

(700, 17)


,Airline Data[Flight ID],Airline Data[Flight Number],Airline Data[Airline],Airline Data[Flight Date],Airline Data[Origin City],Airline Data[Origin Airport],Airline Data[Destination City],Airline Data[Destination Airport],Airline Data[Distance (km)],Airline Data[Seats Available],Airline Data[Passengers],Airline Data[Flight Status],Airline Data[Delay (Minutes)],Airline Data[Cancellation Reason],Airline Data[Scheduled Duration (Min)],Airline Data[Revenue (INR)],Airline Data[On Time Flag]
0,21,IX1712,Air India Express,"Sat, 09 Aug 2025 00:00:00",Mumbai,BOM,Kochi,COK,1793,210,178,On Time,0,NaN,145,701854,1
1,22,SG3106,Air India Express,"Mon, 23 Jun 2025 00:00:00",Bengaluru,BLR,Pune,PNQ,1218,186,133,On Time,0,NaN,82,618184,1
2,25,IX8983,Air India Express,"Sun, 12 Oct 2025 00:00:00",Hyderabad,HYD,Pune,PNQ,419,186,109,On Time,0,NaN,179,898596,1
3,27,QP1443,Air India Express,"Thu, 01 May 2025 00:00:00",Mumbai,BOM,Kochi,COK,1517,220,100,On Time,0,NaN,158,382500,1
4,33,UK2747,Air India Express,"Tue, 05 Aug 2025 00:00:00",Bengaluru,BLR,Mumbai,BOM,567,180,81,On Time,0,NaN,125,706239,1


In [4]:
df.columns = [c.split("[")[-1].replace("]", "") for c in df.columns]

print(df.columns.tolist())
print(df["Delay (Minutes)"].describe())

['Flight ID', 'Flight Number', 'Airline', 'Flight Date', 'Origin City', 'Origin Airport', 'Destination City', 'Destination Airport', 'Distance (km)', 'Seats Available', 'Passengers', 'Flight Status', 'Delay (Minutes)', 'Cancellation Reason', 'Scheduled Duration (Min)', 'Revenue (INR)', 'On Time Flag']
count    700.000000
mean       9.942857
std       27.117023
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      180.000000
Name: Delay (Minutes), dtype: float64


In [5]:
print(df["Flight Status"].value_counts())
print(df["On Time Flag"].value_counts())

Flight Status
On Time      525
Delayed      144
Cancelled     31
Name: count, dtype: int64
On Time Flag
1    525
0    175
Name: count, dtype: int64


In [6]:
df["Delayed"] = (df["Delay (Minutes)"] > 0).astype(int)

print(df["Delayed"].value_counts())

Delayed
0    556
1    144
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = df[[
    "Airline", "Origin City", "Destination City",
    "Distance (km)", "Seats Available", "Passengers",
    "Scheduled Duration (Min)"
]]
y = df["Delayed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat = ["Airline", "Origin City", "Destination City"]
num = ["Distance (km)", "Seats Available", "Passengers", "Scheduled Duration (Min)"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
], remainder="passthrough")

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.7928571428571428
              precision    recall  f1-score   support

           0       0.80      0.99      0.88       111
           1       0.50      0.03      0.06        29

    accuracy                           0.79       140
   macro avg       0.65      0.51      0.47       140
weighted avg       0.74      0.79      0.71       140



In [8]:
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.80      1.00      0.89       111
           1       1.00      0.03      0.07        29

    accuracy                           0.80       140
   macro avg       0.90      0.52      0.48       140
weighted avg       0.84      0.80      0.72       140



In [9]:
proba = model.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.4, 0.3, 0.2]:
    pred_t = (proba >= threshold).astype(int)
    print("\nThreshold:", threshold)
    print(classification_report(y_test, pred_t))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.80      1.00      0.89       111
           1       1.00      0.03      0.07        29

    accuracy                           0.80       140
   macro avg       0.90      0.52      0.48       140
weighted avg       0.84      0.80      0.72       140


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.80      0.98      0.88       111
           1       0.33      0.03      0.06        29

    accuracy                           0.79       140
   macro avg       0.56      0.51      0.47       140
weighted avg       0.70      0.79      0.71       140


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       111
           1       0.24      0.17      0.20        29

    accuracy                           0.71       140
   macro avg       0.52      0.51      0.51       140
weighted avg       0.68   

In [12]:
 all_proba = model.predict_proba(X)[:, 1]

df["Delay Probability"] = all_proba
df["AI Risk"] = ["High Risk" if p >= 0.3 else "Low Risk" for p in all_proba]

print(df["AI Risk"].value_counts())

AI Risk
Low Risk     564
High Risk    136
Name: count, dtype: int64


In [14]:
df.to_excel("Airline_AI_Predictions.xlsx", index=False)
files.download("Airline_AI_Predictions.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>